## Updated CLI commands

This notebook was updated to remove dependency on `hera.utils.data.cli_toolkit_repository`.

Use the new CLIs instead:
- `hera-toolkit import-json --project <PROJECT> --file <REPOSITORY_JSON_PATH>`
- `hera-toolkit register --project <PROJECT> --cls <PYTHON_CLASSPATH> --name <DS_NAME>`
- `hera-project repository load <REPOSITORY_NAME> <PROJECT> [--overwrite]`

> Replace placeholders like `<PROJECT>` and `<REPOSITORY_JSON_PATH>` with your values.


# Jerusalem2018 Toolkit – Usage Guide

This notebook shows how to:
1. Register the Jerusalem 2018 experiment as a dynamic toolkit.
2. Load it via `DataHandler_Class.getData`.
3. Inspect the class API and discover useful methods.
4. (Optional) call example methods if they exist (placeholders).

> You need the code on disk (e.g., cloned to `~/hera-jerusalem-main`).


In [ ]:

PROJECT = "UnitTestProject"
JER_ROOT = os.path.expanduser("~/hera-jerusalem-main")
JER_CODE = os.path.join(JER_ROOT, "code")
JER_DATA = os.path.join(JER_ROOT, "data")
os.makedirs(JER_DATA, exist_ok=True)
print("Project:", PROJECT)
print("Jerusalem code dir:", JER_CODE)
print("Jerusalem data dir:", JER_DATA)


## 1) Register (two options)
### Option A – CLI

In [ ]:

        print(
            "hera-toolkit import-json --project <PROJECT> --file <REPOSITORY_JSON_PATH> register-datasource \n"
            f"  --project {PROJECT} \
"
            "  --name Jerusalem2018_EXT \
"
            "  --classpath Jerusalem2018.Jerusalem2018 \
"
            f"  --resource {JER_CODE} \
"
            f"  --params '{{"projectName":"{PROJECT}","pathToExperiment":"{JER_ROOT}","filesDirectory":"{JER_DATA}"}}' \
"
            "  --version 0.0.1 \
"
            "  --overwrite"
        )


### Option B – Python API

In [ ]:

import sys, importlib, os
from hera.toolkit import ToolkitHome

if JER_CODE not in sys.path:
    sys.path.insert(0, JER_CODE)
JerCls = getattr(importlib.import_module("Jerusalem2018"), "Jerusalem2018")

th = ToolkitHome()
doc = th.registerToolkit(
    toolkitclass=JerCls,
    projectName=PROJECT,
    datasource_name="Jerusalem2018_EXT",
    params={"projectName": PROJECT, "pathToExperiment": JER_ROOT, "filesDirectory": JER_DATA},
    version=(0, 0, 1),
    overwrite=True,
)
print("Registered Jerusalem2018_EXT. Resource:", doc.resource)


## 2) Discover toolkits

In [ ]:

from hera.toolkit import ToolkitHome
th = ToolkitHome()
th.getToolkitTable(PROJECT)


## 3) Load the Jerusalem object

In [ ]:

from hera.datalayer import Project
from hera.datalayer.datahandler import DataHandler_Class

p = Project(projectName=PROJECT)
docs = p.getMeasurementsDocuments(type="ToolkitDataSource", datasourceName="Jerusalem2018_EXT")
assert docs, "Jerusalem2018_EXT datasource not found. Did you register it above?"
doc = docs[0]

jer = DataHandler_Class.getData(resource=doc.resource, desc=doc.desc)
print("Loaded:", jer, "| class:", type(jer).__name__)


## 4) Explore public methods and attributes

In [ ]:

import inspect, types, pandas as pd

cls = jer.__class__
print("Source file:", inspect.getsourcefile(cls))

# Methods on class (without touching instance properties that may KeyError)
attrs = []
for name, value in inspect.getmembers(cls):
    if name.startswith("_"):
        continue
    if inspect.isfunction(value) or inspect.ismethoddescriptor(value):
        attrs.append(name)
print("Public methods on class (safe):")
print("\n".join(sorted(attrs)) or "(none)")

# Show a few instance attributes (best-effort)
print("\nInstance attributes (best-effort):")
candidates = ["filesDirectory", "setup", "entitiesTable", "trialsTableAllSets"]
for c in candidates:
    try:
        v = getattr(jer, c)
        print(f" - {c}: type={type(v).__name__}")
    except Exception as e:
        print(f" - {c}: <unavailable: {e}>")


## 5) Example: Trials table (if available)
If the toolkit exposes `trialsTable(trialsetName)` or similar, you can use it like this (edit to your dataset):

In [ ]:

try:
    # Change 'default' to your actual trial-set name if needed:
    tt = jer.trialsTable("default")
    display(tt.head(10))
except Exception as e:
    print("trialsTable('default') not available or failed:", e)


## 6) Example: Fetching data from a trial (if supported)
Some experiments expose `getDataFromTrial(deviceType, deviceName, startTime, endTime, withMetadata=True)`.
This is an example—edit device type/name/time window to match your project.

In [ ]:

try:
    df_trial = jer.getDataFromTrial(
        deviceType="meteo",      # adjust to your schema
        deviceName=None,         # or 'Station_A'
        startTime=None, endTime=None,
        withMetadata=True
    )
    display(df_trial.head(10))
except Exception as e:
    print("getDataFromTrial example failed:", e)


---
**Tip:** After you explore which methods are stable in your local copy, adapt the examples above to your exact workflow.